# ECG Insight - Google Colab PTB-XL Training Workflow

This notebook trains the first ECG Insight baseline CNN on a small real PTB-XL benchmark subset. It is designed for Google Colab with a T4 GPU and saves all durable outputs to Google Drive.

Outputs include `best_checkpoint.pt`, `model.pt`, `model.onnx`, `metrics.json`, confusion matrices, TensorBoard logs, sample predictions, and Grad-CAM-style 1D ECG visualizations.

Clinical safety note: this is an engineering benchmark only. It is not clinically validated and must not be used for patient care.

In [ ]:
# Install dependencies. Colab already includes a CUDA PyTorch build on GPU runtimes.
!pip -q install wfdb scikit-learn onnx onnxruntime tensorboard matplotlib tqdm pandas numpy

import ast
import json
import math
import os
import random
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, multilabel_confusion_matrix, precision_recall_fscore_support, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
import wfdb

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Mount Google Drive for checkpoints, logs, exported models, metrics, and visualizations.
from google.colab import drive
drive.mount('/content/drive')

RUN_NAME = 'ecg-insight-ptbxl-' + datetime.utcnow().strftime('%Y%m%d-%H%M%S')
DRIVE_ROOT = Path('/content/drive/MyDrive/ecg-insight-training')
RUN_DIR = DRIVE_ROOT / RUN_NAME
DATA_DIR = RUN_DIR / 'data'
LOG_DIR = RUN_DIR / 'tensorboard'
ARTIFACT_DIR = RUN_DIR / 'artifacts'
FIGURE_DIR = RUN_DIR / 'figures'
for path in [DATA_DIR, LOG_DIR, ARTIFACT_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_DIR)

In [ ]:
# Training configuration. Defaults are intentionally modest and fit comfortably on a Colab T4.
PTBXL_VERSION = '1.0.3'
PTBXL_METADATA_URL = f'https://physionet.org/files/ptb-xl/{PTBXL_VERSION}/ptbxl_database.csv'
LABELS = ['normal_ecg', 'atrial_fibrillation', 'left_bundle_branch_block', 'right_bundle_branch_block', 'myocardial_infarction']
SCP_TO_LABEL = {
    'NORM': 'normal_ecg',
    'AFIB': 'atrial_fibrillation',
    'CLBBB': 'left_bundle_branch_block',
    'CRBBB': 'right_bundle_branch_block',
    'AMI': 'myocardial_infarction',
    'IMI': 'myocardial_infarction',
    'ASMI': 'myocardial_infarction',
    'ILMI': 'myocardial_infarction',
    'LMI': 'myocardial_infarction',
    'PMI': 'myocardial_infarction',
}

MAX_RECORDS = 300
MAX_PER_CLASS = 80
MAX_SAMPLES = 1000
BATCH_SIZE = 32
EPOCHS = 8
PATIENCE = 3
LEARNING_RATE = 1e-3
THRESHOLD = 0.5

config = {
    'ptbxl_version': PTBXL_VERSION,
    'labels': LABELS,
    'max_records': MAX_RECORDS,
    'max_per_class': MAX_PER_CLASS,
    'max_samples': MAX_SAMPLES,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'learning_rate': LEARNING_RATE,
    'threshold': THRESHOLD,
    'seed': SEED,
}
(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
config

In [ ]:
# Download PTB-XL metadata, select a balanced small benchmark subset, and load waveforms from PhysioNet.
def labels_for_scp(scp_codes):
    parsed = ast.literal_eval(scp_codes) if isinstance(scp_codes, str) else {}
    labels = sorted({SCP_TO_LABEL[code] for code in parsed if code in SCP_TO_LABEL})
    return labels

def normalize_signal(signal):
    signal = signal.astype(np.float32).T
    if signal.shape[0] != 12:
        raise ValueError(f'Expected 12 leads, got {signal.shape[0]}')
    if signal.shape[1] > MAX_SAMPLES:
        signal = signal[:, :MAX_SAMPLES]
    elif signal.shape[1] < MAX_SAMPLES:
        pad = MAX_SAMPLES - signal.shape[1]
        signal = np.pad(signal, ((0, 0), (0, pad)), mode='constant')
    mean = signal.mean(axis=1, keepdims=True)
    std = signal.std(axis=1, keepdims=True)
    return (signal - mean) / np.maximum(std, 1e-6)

metadata = pd.read_csv(PTBXL_METADATA_URL)
metadata['target_labels'] = metadata['scp_codes'].apply(labels_for_scp)
metadata = metadata[metadata['target_labels'].map(len) > 0].sample(frac=1, random_state=SEED).reset_index(drop=True)

selected_rows = []
class_counts = {label: 0 for label in LABELS}
for _, row in metadata.iterrows():
    row_labels = row['target_labels']
    if any(class_counts[label] < MAX_PER_CLASS for label in row_labels):
        selected_rows.append(row)
        for label in row_labels:
            class_counts[label] += 1
    if len(selected_rows) >= MAX_RECORDS or all(count >= MAX_PER_CLASS for count in class_counts.values()):
        break

signals = []
targets = []
record_ids = []
manifest_rows = []
for index, row in enumerate(tqdm(selected_rows, desc='Downloading PTB-XL records')):
    record_path = row['filename_lr'].replace('.dat', '')
    record = wfdb.rdrecord(record_path, pn_dir=f'ptb-xl/{PTBXL_VERSION}')
    signal = normalize_signal(record.p_signal)
    label_vector = np.array([1.0 if label in row['target_labels'] else 0.0 for label in LABELS], dtype=np.float32)
    signals.append(signal)
    targets.append(label_vector)
    record_ids.append(str(row['ecg_id']))
    manifest_rows.append({'ecg_id': row['ecg_id'], 'filename_lr': row['filename_lr'], 'labels': '|'.join(row['target_labels'])})

X = np.stack(signals).astype(np.float32)
y = np.stack(targets).astype(np.float32)
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(DATA_DIR / 'ptbxl_colab_manifest.csv', index=False)
np.savez_compressed(DATA_DIR / 'ptbxl_colab_subset.npz', X=X, y=y, record_ids=np.array(record_ids), labels=np.array(LABELS))
print('Dataset shape:', X.shape, y.shape)
print('Class counts:', dict(zip(LABELS, y.sum(axis=0).astype(int).tolist())))

In [ ]:
# Create 70% train, 15% validation, 15% test splits.
indices = np.arange(len(X))
rng = np.random.default_rng(SEED)
rng.shuffle(indices)
train_end = int(0.70 * len(indices))
validation_end = int(0.85 * len(indices))
splits = {
    'train': indices[:train_end],
    'validation': indices[train_end:validation_end],
    'test': indices[validation_end:],
}
split_summary = {name: int(len(values)) for name, values in splits.items()}
(RUN_DIR / 'split_summary.json').write_text(json.dumps(split_summary, indent=2), encoding='utf-8')

def loader_for(split_name, shuffle=False):
    split_indices = splits[split_name]
    dataset = TensorDataset(torch.tensor(X[split_indices]), torch.tensor(y[split_indices]))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=(device.type == 'cuda'))

train_loader = loader_for('train', shuffle=True)
validation_loader = loader_for('validation')
test_loader = loader_for('test')
split_summary

In [ ]:
# Baseline ECG CNN model.
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),
        )
    def forward(self, x):
        return self.block(x)

class BaselineECGCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = ConvBlock(12, 32)
        self.conv2 = ConvBlock(32, 64)
        self.conv3 = ConvBlock(64, 128)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(128, num_classes))
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.pool(x)
        return self.classifier(x)

model = BaselineECGCNN(num_classes=len(LABELS)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
writer = SummaryWriter(log_dir=str(LOG_DIR))
model

In [ ]:
# Train with early stopping and save checkpoints to Google Drive.
def run_epoch(loader, training):
    model.train(training)
    losses = []
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        if training:
            loss.backward()
            optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else math.nan

best_validation_loss = float('inf')
epochs_without_improvement = 0
history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    with torch.no_grad():
        validation_loss = run_epoch(validation_loader, training=False)
    writer.add_scalar('loss/train', train_loss, epoch)
    writer.add_scalar('loss/validation', validation_loss, epoch)
    row = {'epoch': epoch, 'train_loss': train_loss, 'validation_loss': validation_loss}
    history.append(row)
    print(row)
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'validation_loss': validation_loss,
        'labels': LABELS,
        'config': config,
    }
    torch.save(checkpoint, ARTIFACT_DIR / 'checkpoint.pt')
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        epochs_without_improvement = 0
        torch.save(checkpoint, ARTIFACT_DIR / 'best_checkpoint.pt')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Early stopping at epoch', epoch)
            break

writer.flush()
pd.DataFrame(history).to_csv(RUN_DIR / 'training_history.csv', index=False)

In [ ]:
# Evaluate the best checkpoint and generate metrics plus confusion matrices.
checkpoint = torch.load(ARTIFACT_DIR / 'best_checkpoint.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_probabilities = []
all_targets = []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        logits = model(batch_x.to(device, non_blocking=True))
        all_probabilities.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(batch_y.numpy())

y_score = np.vstack(all_probabilities)
y_true = np.vstack(all_targets)
y_pred = (y_score >= THRESHOLD).astype(np.float32)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
per_label_precision, per_label_recall, per_label_f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
try:
    auroc = float(roc_auc_score(y_true, y_score, average='macro'))
except ValueError:
    auroc = None
metrics = {
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'precision_macro': float(precision),
    'recall_macro': float(recall),
    'f1_macro': float(f1),
    'auroc_macro': auroc,
    'labels': LABELS,
    'per_label': {
        label: {
            'precision': float(per_label_precision[i]),
            'recall': float(per_label_recall[i]),
            'f1': float(per_label_f1[i]),
        }
        for i, label in enumerate(LABELS)
    },
}
(ARTIFACT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))

cms = multilabel_confusion_matrix(y_true, y_pred)
confusion_payload = {label: cms[i].tolist() for i, label in enumerate(LABELS)}
(ARTIFACT_DIR / 'confusion_matrices.json').write_text(json.dumps(confusion_payload, indent=2), encoding='utf-8')

fig, axes = plt.subplots(1, len(LABELS), figsize=(4 * len(LABELS), 4))
for i, label in enumerate(LABELS):
    ax = axes[i]
    im = ax.imshow(cms[i], cmap='Blues')
    ax.set_title(label)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1']); ax.set_yticklabels(['True 0', 'True 1'])
    for row in range(2):
        for col in range(2):
            ax.text(col, row, int(cms[i, row, col]), ha='center', va='center')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'confusion_matrices.png', dpi=160)
plt.show()

In [ ]:
# Export model.pt and model.onnx to Google Drive.
model.eval()
torch.save({'model_state_dict': model.state_dict(), 'labels': LABELS, 'config': config}, ARTIFACT_DIR / 'model.pt')
dummy = torch.zeros(1, 12, MAX_SAMPLES, dtype=torch.float32, device=device)
torch.onnx.export(
    model,
    dummy,
    ARTIFACT_DIR / 'model.onnx',
    input_names=['ecg'],
    output_names=['logits'],
    dynamic_axes={'ecg': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
print('Saved:', ARTIFACT_DIR / 'model.pt')
print('Saved:', ARTIFACT_DIR / 'model.onnx')

In [ ]:
# Generate sample predictions from the held-out test split.
sample_predictions = []
test_indices = splits['test'][:10]
with torch.no_grad():
    for rank, original_index in enumerate(test_indices):
        sample = torch.tensor(X[original_index:original_index + 1]).to(device)
        probabilities = torch.sigmoid(model(sample))[0].cpu().numpy()
        sample_predictions.append({
            'rank': int(rank),
            'record_id': record_ids[original_index],
            'true_labels': [LABELS[i] for i, value in enumerate(y[original_index]) if value == 1],
            'probabilities': {LABELS[i]: float(probabilities[i]) for i in range(len(LABELS))},
            'predicted_labels': [LABELS[i] for i, value in enumerate(probabilities) if value >= THRESHOLD],
        })
(ARTIFACT_DIR / 'sample_predictions.json').write_text(json.dumps(sample_predictions, indent=2), encoding='utf-8')
sample_predictions[:3]

In [ ]:
# Grad-CAM-style 1D ECG visualizations for the final convolution block.
def grad_cam_1d(sample_tensor, class_index):
    activations = []
    gradients = []
    def forward_hook(_module, _inputs, output):
        activations.append(output.detach())
    def backward_hook(_module, _grad_input, grad_output):
        gradients.append(grad_output[0].detach())
    handle_fwd = model.conv3.block[0].register_forward_hook(forward_hook)
    handle_bwd = model.conv3.block[0].register_full_backward_hook(backward_hook)
    model.zero_grad(set_to_none=True)
    logits = model(sample_tensor)
    logits[0, class_index].backward()
    handle_fwd.remove(); handle_bwd.remove()
    activation = activations[0][0]
    gradient = gradients[0][0]
    weights = gradient.mean(dim=1)
    cam = torch.relu((weights[:, None] * activation).sum(dim=0)).cpu().numpy()
    cam = np.interp(np.linspace(0, len(cam) - 1, MAX_SAMPLES), np.arange(len(cam)), cam)
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
    return cam

for rank, original_index in enumerate(splits['test'][:3]):
    sample_tensor = torch.tensor(X[original_index:original_index + 1]).to(device)
    probabilities = torch.sigmoid(model(sample_tensor)).detach().cpu().numpy()[0]
    class_index = int(np.argmax(probabilities))
    cam = grad_cam_1d(sample_tensor, class_index)
    lead_ii = X[original_index, 1]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(lead_ii, color='black', linewidth=1.0, label='Lead II')
    ax.imshow(cam[None, :], extent=[0, MAX_SAMPLES, lead_ii.min(), lead_ii.max()], aspect='auto', cmap='jet', alpha=0.28)
    ax.set_title(f"Grad-CAM ECG {record_ids[original_index]} | predicted {LABELS[class_index]} ({probabilities[class_index]:.2f})")
    ax.set_xlabel('Sample')
    ax.set_ylabel('Normalized amplitude')
    ax.legend(loc='upper right')
    fig.tight_layout()
    output_path = FIGURE_DIR / f'grad_cam_{rank}_{record_ids[original_index]}.png'
    fig.savefig(output_path, dpi=160)
    plt.show()
print('Grad-CAM visualizations saved to', FIGURE_DIR)

## Review Outputs

Open the run directory in Google Drive and review:

- `artifacts/model.onnx`
- `artifacts/model.pt`
- `artifacts/best_checkpoint.pt`
- `artifacts/metrics.json`
- `artifacts/confusion_matrices.json`
- `artifacts/sample_predictions.json`
- `figures/confusion_matrices.png`
- `figures/grad_cam_*.png`
- `tensorboard/` logs

For TensorBoard in Colab, run `%load_ext tensorboard` and `%tensorboard --logdir "$LOG_DIR"` in a new cell if interactive monitoring is needed.